# 📓 Notebook 5 — Pandas Preview: Your First Real DataFrame

> **Module:** Data Science Libraries · **Estimated time:** 30–40 min · **Difficulty:** Beginner

You can do real data work in pure Python — that's exactly what you did with lists (Notebook 3) and dicts (Notebook 4). But once your data is more than a handful of rows, you want a **proper table**: row + column labels, fast filtering, group-by operations, reading and writing CSV files. That is **pandas**.

Pandas is *the* library every data scientist touches every day. This notebook gives you a working preview: enough to be productive, with the deep dive coming after NumPy and matplotlib.

In keeping with the course identity, every example in this notebook uses a **business-AI dataset** — a log of LLM API calls with their model, token usage, cost, latency, and customer segment. By the end of the notebook you'll have built a small DataFrame, filtered it, grouped it, and drawn your first plot from it.

## 🎯 Learning objectives

1. Build a `DataFrame` from a dict and from a CSV.
2. Inspect a new dataset with the standard "first five commands".
3. Select columns and rows (`.loc` vs `.iloc`).
4. Filter rows with **boolean masks** — the pandas idiom.
5. Add derived columns from formulas.
6. Use `groupby` to compute aggregates by category.
7. Draw a first plot directly from a DataFrame.

## ✅ Prerequisites

Notebooks 1–4.

## 1. The pandas mental model

Pandas has two core objects:

| Object        | Conceptually          | Compare to                        |
|---------------|-----------------------|-----------------------------------|
| `Series`      | A single column       | a labelled NumPy 1-D array        |
| `DataFrame`   | A whole table         | a labelled 2-D array / SQL table  |

```
   ┌────────────────  DataFrame  ────────────────┐
   │  request_id   model       tokens   cost    │
   │  req_001      gpt-4o-mini  482     0.00029 │
   │  req_002      claude-haiku 612     0.00037 │
   │  ...                                       │
   └────────────────────────────────────────────┘
                ↑                  ↑
               row              column
            (a record)        (a Series)
```

If you understand a list-of-dicts from Notebook 4, you already understand a DataFrame — pandas just gives you a *much* better toolkit for working with it.

In [ ]:
# The standard import aliases — you'll see "pd" and "np" in every pandas notebook ever
import pandas as pd
import numpy as np

print(f"pandas version: {pd.__version__}")
print(f"numpy  version: {np.__version__}")


## 2. Creating a DataFrame from a dict

The most natural way to build a small DataFrame is from a dict where **keys are column names** and **values are lists** (one per row).

In [ ]:
# A small log of LLM API calls from yesterday's batch
data = {
    "request_id":     ["req_001", "req_002", "req_003", "req_004", "req_005", "req_006", "req_007", "req_008"],
    "model":          ["gpt-4o-mini", "gpt-4o-mini", "claude-haiku", "gpt-4o-mini",
                       "claude-haiku", "gpt-4o-mini", "claude-haiku", "gpt-4o-mini"],
    "tokens_in":      [482, 612, 510, 1180, 720,  430,  890, 240],
    "tokens_out":     [121, 158, 144,  205, 192,  118,  201,  87],
    "latency_ms":     [1820, 2150, 1640, 3180, 2410, 1740, 2530, 1380],
    "customer_seg":   ["SMB", "Enterprise", "SMB", "Enterprise",
                       "Mid-market", "SMB", "Enterprise", "SMB"],
}

df = pd.DataFrame(data)
df


**A pandas tip.** Inside Jupyter, just typing the variable name (`df`) at the bottom of a cell renders a beautifully formatted HTML table. Try `print(df)` *and* just `df` — the latter is what you'll prefer.

## 3. Inspecting a DataFrame

Whenever you load a new dataset, run these commands first. They tell you the *shape*, the *types*, and the *typical values* before you do anything else. Skipping this step is one of the most common rookie mistakes.

In [ ]:
print("Shape (rows, cols):", df.shape)
print("\nColumn names :", df.columns.tolist())
print("\nData types:")
print(df.dtypes)


In [ ]:
# The first / last few rows
print("--- head(3) ---")
print(df.head(3))

print("\n--- tail(2) ---")
print(df.tail(2))


In [ ]:
# Quick statistical summary for numeric columns
df.describe()


In [ ]:
# Schema and non-null counts — your missing-data canary
df.info()


## 4. Selecting columns

These three patterns appear in every pandas notebook:

```python
df["tokens_in"]                       # one column → Series
df[["request_id", "tokens_in"]]       # several columns → DataFrame
df.tokens_in                          # attribute-style (only when the name is a valid identifier)
```

In [ ]:
# One column → Series
tokens = df["tokens_in"]
print(tokens)
print(f"\ntype: {type(tokens).__name__}")


In [ ]:
# Several columns → DataFrame
subset = df[["request_id", "model", "tokens_in", "tokens_out"]]
subset


## 5. Selecting rows — `loc` vs `iloc`

Pandas gives you **two** ways to slice rows. The difference matters once your DataFrame has a non-default index.

| Indexer | What it uses          | Endpoint inclusive?   |
|---------|------------------------|-----------------------|
| `.loc`  | **labels** (the index) | yes (`df.loc[0:2]` returns 3 rows) |
| `.iloc` | **integer positions**  | no (`df.iloc[0:2]` returns 2 rows) |

In [ ]:
# .loc — label-based
print("df.loc[0]:")
print(df.loc[0])           # first row → Series

print("\ndf.loc[0:2, ['request_id', 'tokens_in']]:")
print(df.loc[0:2, ["request_id", "tokens_in"]])


In [ ]:
# .iloc — position-based
print("df.iloc[0]:")
print(df.iloc[0])

print("\ndf.iloc[0:2, 0:3]:")
print(df.iloc[0:2, 0:3])    # first 2 rows, first 3 cols


> 🎯 **Rule of thumb.** Prefer `.loc` — it reads almost like English. Use `.iloc` only when you really do mean "the *n*-th row" regardless of labels.

## 6. Filtering — the boolean-mask idiom

This is **the** pattern of pandas: build a Series of `True`/`False` values that says which rows you want, then index the DataFrame with it.

```python
df[ df["tokens_in"] > 500 ]
   └────────────────────┘
   boolean Series the same length as df
```

In [ ]:
# Step 1: build the mask
mask = df["tokens_in"] > 500
print("Mask:")
print(mask)

# Step 2: apply it
print("\nLarge-prompt requests:")
print(df[mask])


In [ ]:
# Combining conditions — note the parentheses around each part, and & (not `and`)
big_and_slow = df[(df["tokens_in"] > 500) & (df["latency_ms"] > 2000)]
print("Big AND slow requests:")
print(big_and_slow)

# OR uses |
enterprise_or_huge = df[(df["customer_seg"] == "Enterprise") | (df["tokens_in"] > 1000)]
print("\nEnterprise OR very long:")
print(enterprise_or_huge)


> ⚠️ Two common stumbles:
> - Use **`&` and `|`**, *not* `and` / `or`, for element-wise boolean operations.
> - **Wrap each condition in parentheses** — `&` and `|` have higher precedence than `>` and `==`.

## 7. Adding and modifying columns

Derived columns are how you turn raw data into business KPIs.

In [ ]:
# A simple price table (you'd normally load this from config)
prices_per_1k = {
    "gpt-4o-mini" : {"in": 0.0006, "out": 0.0024},
    "claude-haiku": {"in": 0.0008, "out": 0.0040},
}

# Map model → price using a small lambda
df["price_in_per_1k"]  = df["model"].map(lambda m: prices_per_1k[m]["in"])
df["price_out_per_1k"] = df["model"].map(lambda m: prices_per_1k[m]["out"])

# Compute the cost of every request — vectorised arithmetic, no loop needed
df["cost_usd"] = (df["tokens_in"]  / 1000 * df["price_in_per_1k"] +
                  df["tokens_out"] / 1000 * df["price_out_per_1k"])

# A boolean column — was this a slow call?
df["was_slow"] = df["latency_ms"] > 2000

# A categorical column with pd.cut — three latency bands
df["latency_band"] = pd.cut(
    df["latency_ms"],
    bins=[0, 1500, 2500, np.inf],
    labels=["fast", "medium", "slow"],
)

df


> 🎯 **Vectorised vs loops.** Notice we did *not* write a `for` loop to compute `cost_usd`. Pandas computes the formula for every row at once. This is faster *and* shorter than a loop — it's why you reach for pandas in the first place.

## 8. Summary statistics & `value_counts`

In [ ]:
print("Latency stats (ms):")
print(f"  mean   : {df['latency_ms'].mean():.0f}")
print(f"  median : {df['latency_ms'].median():.0f}")
print(f"  std    : {df['latency_ms'].std():.0f}")
print(f"  min    : {df['latency_ms'].min()}")
print(f"  max    : {df['latency_ms'].max()}")

print("\nRequests per model:")
print(df["model"].value_counts())

print("\nRequests per customer segment:")
print(df["customer_seg"].value_counts())


## 9. `groupby` — the most useful pandas verb

`groupby` answers questions of the form *"what's the *something* of *something_else*, broken down by *category*?"*. For instance: *"average cost per request, by model"*, *"total tokens, by customer segment"*, *"slow-call rate by model"*.

The mental model is **split → apply → combine**:

```
                    apply (mean, sum, count, …)
                       ↓
   ┌─ gpt-4o-mini : ████  →  $0.0014 / call
df ┤
   └─ claude-haiku: ███   →  $0.0021 / call
                                 ↑
                              combine
```

In [ ]:
# Average cost per request, per model
print("Mean cost per request, by model:")
print(df.groupby("model")["cost_usd"].mean().round(6))

# Multiple aggregations at once, by model
agg = df.groupby("model").agg(
    n_calls         =("request_id", "count"),
    total_tokens    =("tokens_in",  "sum"),
    mean_latency_ms =("latency_ms", "mean"),
    total_cost      =("cost_usd",   "sum"),
).round(4)
agg


In [ ]:
# Grouping by two columns — model and customer segment
cross = df.groupby(["model", "customer_seg"]).agg(
    n          =("request_id", "count"),
    mean_cost  =("cost_usd",   "mean"),
).round(5)
cross


## 10. Reading and writing CSV files

This is how every real project starts and ends. Pandas reads dozens of file formats (CSV, Excel, JSON, Parquet, SQL, …) with a one-line call.

In [ ]:
# Reading from a string is great for self-contained demos.
# In a real notebook this would be:  api_log = pd.read_csv("api_log.csv")

from io import StringIO

csv_text = '''request_id,model,tokens_in,tokens_out,latency_ms,customer_seg
req_009,gpt-4o-mini,520,140,1900,SMB
req_010,claude-haiku,810,210,2380,Enterprise
req_011,gpt-4o-mini,295,82,1420,SMB
req_012,gpt-4o-mini,1050,260,3050,Enterprise
req_013,claude-haiku,640,170,2210,Mid-market
req_014,gpt-4o-mini,470,128,1810,SMB
req_015,claude-haiku,920,235,2670,Enterprise
'''

api_log = pd.read_csv(StringIO(csv_text))
print(api_log)
print(f"\nShape: {api_log.shape}")


In [ ]:
# Group-by on the loaded data: tokens per segment
tokens_by_seg = api_log.groupby("customer_seg")[["tokens_in", "tokens_out"]].sum()
print(tokens_by_seg)

# To save: api_log.to_csv("api_log.csv", index=False)


## 11. A first plot — `df.plot()`

Pandas has a built-in `.plot()` method that wraps matplotlib. It's perfect for quick exploration. We'll do plotting properly in Notebook 8; for now, a glimpse:

In [ ]:
import matplotlib.pyplot as plt

# Total cost per model — a quick bar chart
cost_by_model = df.groupby("model")["cost_usd"].sum().sort_values(ascending=False)

ax = cost_by_model.plot(
    kind="bar",
    color=["#4C72B0", "#DD8452"],
    title="Total cost by model",
    figsize=(7, 4),
    rot=0,
    edgecolor="black",
)
ax.set_ylabel("Total cost (USD)")
ax.grid(axis="y", alpha=0.3)
for x, v in enumerate(cost_by_model.values):
    ax.text(x, v, f"${v:.4f}", ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
# Latency distribution per model — boxplot
fig, ax = plt.subplots(figsize=(7, 4))
df.boxplot(column="latency_ms", by="model", ax=ax)
ax.set_title("Latency distribution per model")
ax.set_ylabel("Latency (ms)")
plt.suptitle("")     # suppress the auto-title pandas adds
plt.tight_layout()
plt.show()


## 12. The pandas patterns you'll see everywhere

Burn these into memory — they form the daily-use vocabulary of every data scientist:

```python
# Loading
df = pd.read_csv("data.csv")

# Exploring
df.head()                       # see the first rows
df.shape                        # rows, cols
df.describe()                   # summary stats
df.info()                       # types and missing counts

# Selecting
df["col"]                       # one column → Series
df[["col1", "col2"]]            # several columns → DataFrame
df.loc[label, "col"]            # row by label, column by name
df.iloc[0, 0]                   # row by position, column by position

# Filtering
df[df["col"] > 100]
df[(df["col1"] > 100) & (df["col2"] == "x")]

# Adding columns
df["new"] = df["a"] * df["b"]
df["band"] = pd.cut(df["score"], bins=..., labels=...)

# Aggregating
df.groupby("category")["value"].mean()
df.groupby("category").agg(total=("value", "sum"))

# Quick visual
df["amount"].plot(kind="hist", bins=30)
```

Once these feel automatic, the rest of pandas is "more of the same".

## 🧪 Practice exercises

For the next exercises we'll use a slightly larger synthetic dataset: 50 API calls across models, segments, and quarters.

In [ ]:
rng = np.random.default_rng(42)
n = 50

api_data = pd.DataFrame({
    "request_id": [f"req_{i:03d}" for i in range(1, n + 1)],
    "model":      rng.choice(["gpt-4o-mini", "claude-haiku", "gpt-4o"], n),
    "segment":    rng.choice(["SMB", "Mid-market", "Enterprise"], n),
    "quarter":    rng.choice(["Q1", "Q2", "Q3", "Q4"], n),
    "tokens":     rng.integers(120, 2000, n).astype(int),
    "latency_ms": rng.integers(800, 4500, n),
    "user_age":   rng.integers(20, 65, n),
})
api_data.head()


### Exercise 1 — Basic exploration

1. Print the **shape** and **dtypes** of `api_data`.
2. Print the **mean** and **median** of `tokens`.
3. Show how many requests there were per `model`.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
print(f"shape  = {api_data.shape}")
print(f"\ndtypes:\n{api_data.dtypes}")

print(f"\nmean tokens   = {api_data['tokens'].mean():.1f}")
print(f"median tokens = {api_data['tokens'].median():.1f}")

print("\nRequests per model:")
print(api_data['model'].value_counts())
```
</details>

### Exercise 2 — Filtering

Find all requests where **`tokens >= 1500` AND `latency_ms >= 3000`** (the expensive, slow ones). How many are there, and what's their mean latency?

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
expensive_slow = api_data[(api_data["tokens"] >= 1500) & (api_data["latency_ms"] >= 3000)]

print(f"Matching rows : {len(expensive_slow)}")
print(f"Mean latency  : {expensive_slow['latency_ms'].mean():.0f} ms")
expensive_slow.head()
```

These are the calls worth profiling first — biggest cost *and* biggest user-experience hit.
</details>

### Exercise 3 — Group-by

For each `model`, compute:

1. Total `tokens`.
2. Number of requests.
3. The **segment** that consumed the most tokens for that model (think: groupby on two columns).

Tip: `df.groupby(["model", "segment"])["tokens"].sum()` returns a stacked Series — combine with `.idxmax()` or `.sort_values()`.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
# 1 & 2: per-model summary
summary = api_data.groupby("model").agg(
    total_tokens =("tokens", "sum"),
    n_requests   =("tokens", "count"),
)
print(summary)

# 3: top segment per model
grouped = api_data.groupby(["model", "segment"])["tokens"].sum()
print("\nTop segment per model:")
print(grouped.groupby(level=0).idxmax())   # → (model, segment) tuples
```

The pattern `grouped.groupby(level=0).idxmax()` finds, *within each model group*, which segment had the highest total tokens — a one-line "argmax over a subgroup" trick worth remembering.
</details>

### Exercise 4 — Visual exploration

Create two plots:

1. A **bar chart** of total `tokens` per `model`.
2. A **histogram** of `latency_ms` with 10 bins.

Add titles and axis labels.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 1) Bar chart
api_data.groupby("model")["tokens"].sum().plot(
    kind="bar", ax=axes[0], color="#4C72B0", edgecolor="black"
)
axes[0].set_title("Total tokens per model")
axes[0].set_ylabel("Tokens")
axes[0].tick_params(axis="x", rotation=0)
axes[0].grid(axis="y", alpha=0.3)

# 2) Histogram
api_data["latency_ms"].plot(
    kind="hist", bins=10, ax=axes[1], color="#DD8452", edgecolor="black"
)
axes[1].set_title("Latency distribution")
axes[1].set_xlabel("Latency (ms)")

plt.tight_layout()
plt.show()
```
</details>

### Exercise 5 — Debug me 🐞

The cell below should print the **average latency per model** but it raises an error. Fix it.

In [ ]:
# Buggy:
# print(api_data.groupby(model).mean(latency_ms))


<details>
<summary>💡 <b>Solution</b></summary>

Two issues:

1. `model` and `latency_ms` need to be **strings** (column names), not bare identifiers.
2. `.mean(latency_ms)` is not how you pick a column — `.mean()` takes no positional column name.

```python
print(api_data.groupby("model")["latency_ms"].mean().round(0))
```

Or, using `.agg`:

```python
print(api_data.groupby("model").agg(avg_latency=("latency_ms", "mean")).round(0))
```

Both work. The `.agg(...)` form is preferred once you want multiple aggregations at once.
</details>

## 🧠 Stretch exercises

Two more applied exercises to deepen the material. Try them yourself before opening the solution.


### Stretch exercise A — Month-over-month growth in cost

Add a column `cost_growth_mom` to `df` that, for each row, gives the percentage change in `cost_usd` compared with the *previous* row. Sort by `latency_ms` first to make the change meaningful. Print the head.


<details>
<summary>💡 <b>Solution</b></summary>

```python
df_sorted = df.sort_values("latency_ms").copy()
df_sorted["cost_growth_mom"] = df_sorted["cost_usd"].pct_change()
print(df_sorted[["request_id", "latency_ms", "cost_usd", "cost_growth_mom"]].round(4).head())
```

**`pct_change()`** is the pandas one-liner for "change vs previous
row". Same trick on time-indexed data gives day-over-day deltas;
with `freq="M"` it does month-over-month directly. Combine with
`.fillna(0)` if you want the first row to be 0 instead of `NaN`.

</details>

### Stretch exercise B — Pivot table — tokens by model and segment

Build a `pivot_table` showing **mean tokens_in** with `model` as rows and `customer_seg` as columns. Use `margins=True` to add row and column totals.


<details>
<summary>💡 <b>Solution</b></summary>

```python
pivot = df.pivot_table(
    index="model",
    columns="customer_seg",
    values="tokens_in",
    aggfunc="mean",
    margins=True,
    margins_name="all",
).round(0)
print(pivot)
```

**Pivot tables = cross-tabs with aggregation.** Same shape as a
spreadsheet PivotTable; far faster to write. `margins=True` adds
the totals row + column for free — useful for sanity-checking the
cells (they should add up).

</details>

## 🎁 Bonus mini-project — A one-page report on `api_log.csv`

The repository's `data/api_log.csv` file contains 50 LLM API calls in the same
schema you've been working with. Load it from disk and produce a small report:

1. Total tokens consumed (input + output) and an estimated total cost
   using `$0.0006/1K` for input tokens and `$0.0024/1K` for output tokens.
2. The **most expensive call** and which segment it belongs to.
3. The **mean latency** per model.
4. A horizontal bar chart of total cost per model.

Save the enriched DataFrame (with derived `cost_usd` column) back to
`api_log_enriched.csv`. This mini-project mirrors what every
*"please give me a quick summary of yesterday's API spend"* request looks like
in practice.

In [ ]:
# Your code here  👇
# Hint: api_log = pd.read_csv("data/api_log.csv")


<details>
<summary>💡 <b>Solution</b></summary>

```python
api_log = pd.read_csv("data/api_log.csv")

# 1. Total cost using a per-row formula
price_in_per_1k  = 0.0006
price_out_per_1k = 0.0024
api_log["cost_usd"] = (api_log["tokens_in"]  / 1000 * price_in_per_1k +
                       api_log["tokens_out"] / 1000 * price_out_per_1k)

total_cost   = api_log["cost_usd"].sum()
total_tokens = (api_log["tokens_in"] + api_log["tokens_out"]).sum()
print(f"Total tokens : {total_tokens:,}")
print(f"Total cost   : ${total_cost:.4f}")

# 2. Most expensive call
idx_max = api_log["cost_usd"].idxmax()
top = api_log.loc[idx_max]
print(f"\nPriciest call: {top['request_id']} ({top['model']}, "
      f"{top['segment']}) cost ${top['cost_usd']:.5f}")

# 3. Mean latency per model
print("\nMean latency per model (ms):")
print(api_log.groupby('model')['latency_ms'].mean().round(0))

# 4. Plot total cost per model
import matplotlib.pyplot as plt
cost_by_model = api_log.groupby('model')['cost_usd'].sum().sort_values()
ax = cost_by_model.plot(kind='barh', color='#4C72B0', edgecolor='black',
                         figsize=(7, 3.5))
ax.set_title('Total cost per model')
ax.set_xlabel('Cost (USD)')
plt.tight_layout(); plt.show()

# Save enriched data
api_log.to_csv("api_log_enriched.csv", index=False)
```

**Why this matters.** This is the smallest possible end-to-end pandas pipeline:
load → enrich → aggregate → visualise → save. The full capstone in Notebook 10
uses the same five steps on a richer dataset.
</details>

## 🧠 Key takeaways

1. A **DataFrame** is a labelled table; a **Series** is one column.
2. Use `df.head() / .shape / .dtypes / .info() / .describe()` to inspect a new dataset.
3. Select columns with `df["col"]` or `df[[col1, col2]]`; rows with `.loc[label]` or `.iloc[position]`.
4. **Boolean masks** drive filtering: `df[mask]`. Combine masks with `&`, `|`, `~`, and wrap each condition in parentheses.
5. **`groupby` → aggregation** is the workhorse of analysis (`mean`, `sum`, `count`, custom funcs).
6. **Derived columns** turn raw data into KPIs — vectorised arithmetic, no loops.
7. `df.plot()` gets you to a chart in one line.
8. `pd.read_csv` / `df.to_csv` is the universal I/O — almost every project starts and ends there.

## ✅ Self-assessment

- [ ] Build a DataFrame from a dict-of-lists
- [ ] Inspect a new dataset with `head()`, `shape`, `dtypes`, `info()`, `describe()`
- [ ] Select columns and rows with `.loc` and `.iloc`
- [ ] Filter rows with a boolean mask combining multiple conditions
- [ ] Add a derived column from a formula
- [ ] Use `groupby(...).agg(...)` to compute multiple aggregates per group
- [ ] Read a CSV and write one back

## 🚀 Next step

Continue with **Notebook 6 — Functions and Modules**, where you'll learn to package the snippets you've been writing into reusable, named building blocks — the foundation of every clean analysis script.